# RSNA mammography preprocessing: a patient-level MLO cohort in 512 × 512 grayscale

## Objective and scope

This notebook converts the source RSNA mammography archive into the canonical real-image dataset used by the downstream augmentation, generation, and classification experiments. The processing policy is explicit: retain only mediolateral oblique (MLO) views, derive each patient's status as the maximum image-level `cancer` value, exclude negative images belonging to positive patients, select at most one image per patient, and cap the negative-to-positive ratio at 5:1 before partitioning the cohort.

The setup cells below locate the repository and import the libraries required by both execution modes. They first inspect the raw source dataset. If it is unavailable, they perform a read-only audit of the canonical processed cohort and reuse it only when all manifests reconcile, every expected split and class is represented, patient and output paths are unique, and every recorded image exists. If neither representation is complete, the user must provide the original RSNA archive locally; the notebook does not download a dataset or install download helpers automatically. The analysis is configured for 512 × 512 output, a fixed random seed of 42, 15% validation and 15% test fractions, and `RESET_OUTPUT_DIR=False` by default.

**Inputs.** In source-processing mode, the input is `data/original/dataset/train.csv` together with its PNG files, and the table must provide `patient_id`, `image_id`, `laterality`, `view`, and `cancer`. In local-reuse mode, the input is the already materialized `data/processed/metadata/all_processed.csv`, its three split manifests, and the images referenced by `processed_path`.

**Outputs.** Processed images are written to `data/processed/<split>/<label>/`; cohort manifests are written to `data/processed/metadata/`; figures, evaluation records, and sustainability records are written beneath `results/1_preprocessing/01_rsna_512_gray_mlo/`. Images remain single-channel on disk; any later conversion to three channels is intentionally deferred to model-specific loaders.

**Reproducibility and safeguards.** Repository and override paths are validated, a user-supplied archive must be non-empty, and extraction must yield a recognizable dataset. Reuse is never inferred from a directory alone: a dedicated, read-only audit must establish manifest and image completeness before raw-data acquisition is bypassed. Destructive reset is controlled by explicit flags, while validated recovery may replace an incomplete extraction before rebuilding it. Subsequent sections add schema, class, patient-uniqueness, file-existence, and split-disjointness checks. This notebook defines the real cohort used by later experiments without making a model-performance claim.


In [ ]:
# === Bootstrap unificato notebooks/ ===
# Funziona dalla root del progetto e da ogni sottocartella della struttura notebooks/.
import sys as _sys
from pathlib import Path as _Path


def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("Root MammoDiffusion non trovata da " + str(_Path.cwd()))


PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

BASE = PROJECT_ROOT
BASE_DIR = PROJECT_ROOT
BASE_PATH = str(PROJECT_ROOT) + "/"
# === Fine bootstrap unificato ===

from pathlib import Path
from datetime import datetime
import json
import shutil
import sys
import subprocess

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm import tqdm

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

In [ ]:
# Configurazione iniziale e archivio RSNA fornito dall'utente

PROJECT_NAME = "MammoDiffusion"

# Su Colab/Drive impostare ad esempio:
# PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/MammoDiffusion"
PROJECT_ROOT_OVERRIDE = None


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    """Trova automaticamente la root del progetto MammoDiffusion."""
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
        return root

    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name:
            return candidate
        has_notebooks = (candidate / "notebooks").exists() or (candidate / "notebooks").exists()
        if ((candidate / "data").exists() and has_notebooks) or ((candidate / ".git").exists() and has_notebooks):
            return candidate

    # ulteriore check in possibili cartelle
    for candidate in [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
        Path.home() / "Progetti" / project_name,
    ]:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Non riesco a trovare la root del progetto MammoDiffusion.\n"
        "Esegui il notebook dalla repo clonata oppure imposta PROJECT_ROOT_OVERRIDE."
    )


BASE_DIR = find_project_root()
NOTEBOOKS_DIR = BASE_DIR / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

try:
    from eco_tracker import measure_sustainability
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "codecarbon", "psutil"])
    from eco_tracker import measure_sustainability

from processed_dataset_reuse import audit_processed_dataset

ARCHIVE_PATH = (BASE_DIR / "data" / "original" / "dataset.rar").resolve()
EXTRACT_DIR  = (BASE_DIR / "data" / "original" / "_dataset_extract_tmp").resolve()
FINAL_DATASET_DIR = (BASE_DIR / "data" / "original" / "dataset").resolve()

DATASET_DIR = FINAL_DATASET_DIR

# Force source acquisition only when intentionally rebuilding from raw data.
FORCE_REDOWNLOAD_DATASET = False

# Permit a read-only fallback when the raw archive is absent but the canonical
# processed cohort passes every completeness check. Raw data always takes
# precedence when available, preserving the original preprocessing behavior.
ALLOW_COMPLETE_PROCESSED_REUSE = True
PROCESSED_DATA_DIR = (BASE_DIR / "data" / "processed").resolve()

# crea le cartelle necessarie se non esistono
ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)


def dataset_is_ready():
    """
    Controlla se il dataset è già stato estratto correttamente.
    Deve esistere train.csv e deve esserci almeno una PNG.
    """
    if not FINAL_DATASET_DIR.exists():
        return False

    if not (FINAL_DATASET_DIR / "train.csv").exists():
        return False

    png_count = (
        len(list(FINAL_DATASET_DIR.rglob("*.png"))) +
        len(list(FINAL_DATASET_DIR.rglob("*.PNG")))
    )

    return png_count > 0


def prepare_source_dataset():
    """Require a user-supplied source archive; never perform an opaque download."""
    if ARCHIVE_PATH.is_file() and ARCHIVE_PATH.stat().st_size > 0:
        print(f'Using user-supplied RSNA archive: {ARCHIVE_PATH}')
        return
    raise FileNotFoundError(
        'The original RSNA source is not available locally. Obtain it under the official '
        'RSNA Screening Mammography Breast Cancer Detection terms, place the archive at '
        f'{ARCHIVE_PATH}, then rerun. Automatic Google Drive download is disabled.'
    )


def run_command(command):
    # esegue un comando catturando output ed errori e restituendoli come testo
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Comando fallito:\n"
            f"{' '.join(command)}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    return result


def extract_rar_archive():
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    print("Estrazione dataset.rar...")

    # Metodo 1: patool, se trova un estrattore compatibile nel sistema.
    try:
        try:
            import patoolib
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "patool"])
            import patoolib

        patoolib.extract_archive(
            str(ARCHIVE_PATH),
            outdir=str(EXTRACT_DIR)
        )

        print("Estrazione completata con patool.")
        return

    except Exception as e:
        print("patool non è riuscito a estrarre il .rar.")
        print("Errore:", e)

    # Possibili percorsi di 7-Zip su Windows
    possible_7zip_paths = [
        "7z",
        "7zz",
        r"C:\Program Files\7-Zip\7z.exe",
        r"C:\Program Files (x86)\7-Zip\7z.exe",
    ]

    commands_to_try = []

    for seven_zip_path in possible_7zip_paths:
        commands_to_try.append([
            seven_zip_path,
            "x",                                # estrae mantenendo la struttura delle cartelle
            str(ARCHIVE_PATH),                  # archivio da estrarre
            f"-o{EXTRACT_DIR}",                 # directory di destinazione
            "-y"                                # risponde automaticamente "si" alle richieste di sovrascrittura
        ])

    # Altri estrattori possibili
    commands_to_try.extend([
        ["unrar", "x", "-o+", str(ARCHIVE_PATH), str(EXTRACT_DIR)],
        ["tar", "-xf", str(ARCHIVE_PATH), "-C", str(EXTRACT_DIR)],
    ])

    last_errors = []

    for command in commands_to_try:
        try:
            run_command(command)
            print(f"Estrazione completata con: {command[0]}")
            return
        except Exception as e:
            last_errors.append(str(e))

    raise RuntimeError(
        "Non sono riuscito a estrarre dataset.rar.\n\n"
        "Soluzioni consigliate:\n"
        "1. Installa 7-Zip da https://www.7-zip.org/.\n"
        "2. Oppure converti dataset.rar in dataset.zip su Google Drive.\n"
        "3. Oppure, se sei su Colab/Linux, esegui prima: !apt-get install -y unrar\n\n"
        "Ultimo errore:\n"
        f"{last_errors[-1] if last_errors else 'Nessun errore disponibile'}"
    )


def normalize_extracted_dataset_folder():
    """
    Trova automaticamente train.csv dentro l'archivio estratto
    e porta la cartella corretta in BASE_DIR/dataset.
    """

    train_csv_candidates = list(EXTRACT_DIR.rglob("train.csv"))

    if len(train_csv_candidates) == 0:
        raise FileNotFoundError(
            f"Non ho trovato train.csv dentro l'archivio estratto: {EXTRACT_DIR}"
        )

    if len(train_csv_candidates) > 1:
        print("Attenzione: trovati più train.csv. Uso il primo:")
        for candidate in train_csv_candidates:
            print(candidate)

    source_csv = train_csv_candidates[0].resolve()
    source_dataset_dir = source_csv.parent.resolve()

    print("train.csv trovato in:")
    print(source_csv)

    print("Cartella dataset sorgente:")
    print(source_dataset_dir)

    if FINAL_DATASET_DIR.exists():
        shutil.rmtree(FINAL_DATASET_DIR)

    shutil.move(
        str(source_dataset_dir),
        str(FINAL_DATASET_DIR)
    )

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)

    final_csv = FINAL_DATASET_DIR / "train.csv"

    if not final_csv.exists():
        raise FileNotFoundError(f"train.csv non trovato dopo la normalizzazione: {final_csv}")

    png_count = (
        len(list(FINAL_DATASET_DIR.rglob("*.png"))) +
        len(list(FINAL_DATASET_DIR.rglob("*.PNG")))
    )

    if png_count == 0:
        raise FileNotFoundError(f"Nessuna immagine PNG trovata dentro: {FINAL_DATASET_DIR}")

    print("\nDataset pronto.")
    print("FINAL_DATASET_DIR:", FINAL_DATASET_DIR)
    print("train.csv:", final_csv)
    print("PNG trovate:", png_count)


RAW_DATASET_READY = dataset_is_ready()
PROCESSED_REUSE_AUDIT = audit_processed_dataset(
    project_root=BASE_DIR,
    processed_dir=PROCESSED_DATA_DIR,
)
PROCESSED_REUSE_MANIFEST = Path(PROCESSED_REUSE_AUDIT["manifest_path"])
REUSE_EXISTING_PROCESSED_DATA = bool(
    ALLOW_COMPLETE_PROCESSED_REUSE
    and not FORCE_REDOWNLOAD_DATASET
    and not RAW_DATASET_READY
    and PROCESSED_REUSE_AUDIT["ready"]
)

if FORCE_REDOWNLOAD_DATASET:
    if FINAL_DATASET_DIR.exists():
        shutil.rmtree(FINAL_DATASET_DIR)

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)

if REUSE_EXISTING_PROCESSED_DATA:
    print("Raw RSNA source is unavailable; using the verified processed cohort in read-only mode.")
    print("Processed manifest:", PROCESSED_REUSE_MANIFEST)
    print("Rows:", PROCESSED_REUSE_AUDIT["row_count"])
    print("Split counts:", PROCESSED_REUSE_AUDIT["split_counts"])
    print("Label counts:", PROCESSED_REUSE_AUDIT["label_counts"])
    print("Missing processed images:", PROCESSED_REUSE_AUDIT["missing_image_count"])
elif FORCE_REDOWNLOAD_DATASET or not RAW_DATASET_READY:
    if PROCESSED_REUSE_AUDIT["reasons"]:
        print("Processed-data reuse was rejected:")
        for reason in PROCESSED_REUSE_AUDIT["reasons"]:
            print(" -", reason)
    prepare_source_dataset()
    extract_rar_archive()
    normalize_extracted_dataset_folder()
    RAW_DATASET_READY = dataset_is_ready()
else:
    print("Raw dataset is ready; source-archive preparation is not required.")
    print("DATASET_DIR:", DATASET_DIR)
    print("train.csv:", DATASET_DIR / "train.csv")

In [ ]:
# Configurazione preprocessing

CSV_PATH = DATASET_DIR / "train.csv"

OUTPUT_DIR = (BASE_DIR / "data" / "processed").resolve()

RESULTS_DIR = BASE_DIR / "results"
PREPROCESSING_RESULTS_DIR = RESULTS_DIR / "1_preprocessing/01_rsna_512_gray_mlo"
PLOTS_DIR = PREPROCESSING_RESULTS_DIR / "plots"
METRICS_DIR = PREPROCESSING_RESULTS_DIR / "metrics"
ECOTRACKER_DIR = PREPROCESSING_RESULTS_DIR / "ecotracker"
PREPROCESSING_EVALUATION_PATH = METRICS_DIR / "preprocessing_evaluation.json"
PREPROCESSING_ECOTRACKER_PATH = ECOTRACKER_DIR / "preprocessing_ecotracker.json"

IMG_SIZE = 512
RANDOM_STATE = 42

VAL_SIZE = 0.15
TEST_SIZE = 0.15

VIEW_FILTER = "MLO"
IMAGE_EXTENSIONS = [".png", ".PNG"]

# Set only for an intentional rebuild from raw data. It is incompatible
# with read-only processed-data reuse.
RESET_OUTPUT_DIR = False

print("BASE_DIR:", BASE_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("CSV_PATH:", CSV_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PREPROCESSING_RESULTS_DIR:", PREPROCESSING_RESULTS_DIR)

if REUSE_EXISTING_PROCESSED_DATA:
    if RESET_OUTPUT_DIR:
        raise ValueError(
            "RESET_OUTPUT_DIR cannot be enabled while reusing the only complete processed cohort."
        )
    assert PROCESSED_REUSE_AUDIT["ready"], PROCESSED_REUSE_AUDIT["reasons"]
else:
    assert DATASET_DIR.exists(), f"Source dataset directory not found: {DATASET_DIR}"
    assert CSV_PATH.exists(), f"Source CSV not found: {CSV_PATH}"

In [ ]:
# Helper condivisi per le visualizzazioni EDA
for directory in [PLOTS_DIR, METRICS_DIR, ECOTRACKER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EDA_SPLIT_ORDER = ["train", "val", "test"]
EDA_LABEL_ORDER = [0, 1]
EDA_LABEL_NAMES = {0: "negative", 1: "positive"}
EDA_LABEL_COLORS = {0: "#4c78a8", 1: "#e45756"}


def resolve_path(rel_or_abs, base=BASE_DIR):
    p = Path(rel_or_abs)
    return p if p.is_absolute() else base / p


def show_and_save_preprocessing_plot(fig, filename):
    output_path = PLOTS_DIR / filename
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    print(f"Salvato: {output_path}")
    plt.show()
    plt.close(fig)


def set_annotated_bar_ylim(ax, values, top_margin=0.15):
    values_array = np.asarray(values, dtype=float).ravel()
    y_max = float(values_array.max()) if values_array.size else 0.0
    ax.set_ylim(0, y_max * (1.0 + top_margin) if y_max > 0 else 1.0)


def annotate_bars(ax, bars, fmt="{:.0f}"):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            fmt.format(height),
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
        )


def read_grayscale_image_from_row(row):
    image_path = resolve_path(row["processed_path"])
    with Image.open(image_path) as img:
        return np.array(img.convert("L")), image_path


def get_split_dataframe_for_eda():
    required_cols = {"split", "label"}

    if "final_df" in globals() and required_cols.issubset(final_df.columns):
        return final_df.copy(), "final_df"

    csv_path = OUTPUT_DIR / "metadata" / "all_processed.csv"
    if not csv_path.is_file():
        print(f"EDA non eseguita: non trovo final_df né {csv_path}")
        return None, None

    df = pd.read_csv(csv_path).copy()
    missing_cols = required_cols.difference(df.columns)
    if missing_cols:
        print(f"EDA non eseguita: colonne mancanti in {csv_path}: {sorted(missing_cols)}")
        return None, None

    return df, str(csv_path)


def get_processed_dataframe_for_eda():
    required_cols = {"split", "label", "processed_path"}

    if "processed_df" in globals() and required_cols.issubset(processed_df.columns):
        return processed_df.copy(), "processed_df"

    if "processed_rows" in globals() and processed_rows:
        df = pd.DataFrame(processed_rows)
        if required_cols.issubset(df.columns):
            return df, "processed_rows"

    csv_path = OUTPUT_DIR / "metadata" / "all_processed.csv"
    if not csv_path.is_file():
        print(f"EDA non eseguita: metadata preprocessati non trovati: {csv_path}")
        return None, None

    df = pd.read_csv(csv_path).copy()
    missing_cols = required_cols.difference(df.columns)
    if missing_cols:
        print(f"EDA non eseguita: colonne mancanti in {csv_path}: {sorted(missing_cols)}")
        return None, None

    return df, str(csv_path)


def plot_class_distribution(neg_count, pos_count, title, filename,
                            count_unit="immagini", subtitle=None):
    """Bar chart negative vs positive con conteggio e percentuale annotati.
    neg_count/pos_count: interi gia' calcolati a monte (nessun ricalcolo qui).
    count_unit: 'immagini' o 'pazienti' - dichiarato sull'asse per chiarezza.
    """
    counts = {0: int(neg_count), 1: int(pos_count)}
    total = counts[0] + counts[1]
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(
        [EDA_LABEL_NAMES[label] for label in EDA_LABEL_ORDER],
        [counts[label] for label in EDA_LABEL_ORDER],
        color=[EDA_LABEL_COLORS[label] for label in EDA_LABEL_ORDER],
    )
    set_annotated_bar_ylim(ax, [counts[label] for label in EDA_LABEL_ORDER])
    for bar, label in zip(bars, EDA_LABEL_ORDER):
        count = counts[label]
        pct = (count / total * 100) if total else 0.0
        ax.annotate(
            f"{count:,}\n({pct:.1f}%)".replace(",", "."),
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center", va="bottom", fontsize=11,
        )
    ax.set_title(title, fontsize=13, pad=24)
    if subtitle:
        ax.text(0.5, 1.005, subtitle, transform=ax.transAxes,
                ha="center", va="bottom", fontsize=9, color="#555555")
    ax.set_ylabel(f"Numero di {count_unit}")
    ax.grid(axis="y", alpha=0.25)
    show_and_save_preprocessing_plot(fig, filename)


def plot_negative_reduction_funnel(stage_labels, neg_values, pos_values,
                                   filename, count_unit="immagini"):
    """Grafico a cascata: mostra il crollo dei negativi attraverso le tappe,
    con i positivi affiancati per riferimento. Asse Y unico => il calo e' visibile."""
    x = np.arange(len(stage_labels))
    width = 0.38
    fig, ax = plt.subplots(figsize=(10, 5.5))
    neg_bars = ax.bar(x - width / 2, neg_values, width,
                      label="negative", color=EDA_LABEL_COLORS[0])
    pos_bars = ax.bar(x + width / 2, pos_values, width,
                      label="positive", color=EDA_LABEL_COLORS[1])
    set_annotated_bar_ylim(ax, list(neg_values) + list(pos_values))
    for bar in list(neg_bars) + list(pos_bars):
        height = bar.get_height()
        ax.annotate(
            f"{int(height):,}".replace(",", "."),
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3), textcoords="offset points",
            ha="center", va="bottom", fontsize=9,
        )
    ax.set_title("Riduzione progressiva del dataset: negativi vs positivi", fontsize=13)
    ax.set_ylabel(f"Numero di {count_unit}")
    ax.set_xticks(x)
    ax.set_xticklabels(stage_labels)
    ax.legend(title="Classe")
    ax.grid(axis="y", alpha=0.25)
    show_and_save_preprocessing_plot(fig, filename)

## 1. Load and validate the source metadata

In source-processing mode, the following cells read `CSV_PATH` into `df_all`, display its shape, leading rows, and column names, and then enforce the minimum schema required by the curation protocol. In verified local-reuse mode, the same schema fields are loaded from the canonical processed manifest; final-only fields remain available for downstream validation, while derived `label` and `patient_label` columns are recomputed where required to exercise the cohort invariants without claiming access to the unavailable raw population. Patient and image identifiers, view, and laterality are normalized to strings, while `cancer` is converted to an integer label.

**Input:** the source `train.csv` resolved during setup, or the audited canonical processed manifest when the raw source is absent. **In-memory output:** a type-normalized `df_all` table. The printed image-level class counts, number of distinct patients, number of patients with at least one positive image, and view distribution provide an auditable description of the source data. Execution stops with a `ValueError` if any required column is absent, preventing later filters from operating on an incomplete schema. These checks establish the denominator and labeling fields used throughout the remainder of the preprocessing analysis.


In [ ]:
if REUSE_EXISTING_PROCESSED_DATA:
    reuse_manifest_df = pd.read_csv(PROCESSED_REUSE_MANIFEST)
    # Recompute derived cohort labels below while retaining the recorded split
    # and processed-path fields needed for read-only validation.
    df_all = reuse_manifest_df.drop(columns=["label", "patient_label"], errors="ignore").copy()
    metadata_source_description = "verified canonical processed manifest"
else:
    df_all = pd.read_csv(CSV_PATH)
    metadata_source_description = "raw RSNA train.csv"

print("Metadata source:", metadata_source_description)
print("Metadata shape:", df_all.shape)
display(df_all.head())
print(df_all.columns.tolist())

In [ ]:
required_cols = ["patient_id", "image_id", "laterality", "view", "cancer"]

# controlla se mancano colonne
missing_cols = [col for col in required_cols if col not in df_all.columns]
if missing_cols:
    raise ValueError(f"Errore, mancano alcune colonne obbligatorie: {missing_cols}")

df_all = df_all.copy()

df_all["patient_id"] = df_all["patient_id"].astype(str)
df_all["image_id"] = df_all["image_id"].astype(str)
df_all["view"] = df_all["view"].astype(str)
df_all["laterality"] = df_all["laterality"].astype(str)
df_all["cancer"] = df_all["cancer"].astype(int)

print("Distribuzione originale:")
print(df_all["cancer"].value_counts())

print("\nNumero pazienti originali:", df_all["patient_id"].nunique())
print("Pazienti positivi originali:", int(df_all.groupby("patient_id")["cancer"].max().sum()))

print("\nDistribuzione view:")
print(df_all["view"].value_counts())

### 1.1 Source-cohort class distribution

This descriptive analysis counts image-level `cancer = 0` and `cancer = 1` rows in the complete source table, before view restriction or patient-level selection. It is generated only in source-processing mode; a processed manifest cannot recover the excluded source population, so local-reuse mode reports the omission explicitly instead of presenting curated counts as raw-cohort statistics. The shared plotting helper renders counts and percentages, saves `dataset_full_distribution.png` in the preprocessing plot directory, displays the figure, and closes it to release plotting state.

The input is the validated `df_all` table; the outputs are the two integer counts and a persisted bar chart. Percentages are calculated only for annotation and default to zero if the total count is zero. This figure documents the original class composition on the image—not patient—level and serves as the baseline for the later reduction funnel; it is descriptive and does not by itself justify a causal or performance conclusion.


In [ ]:
# Source-population counts can only be reported when the raw metadata is available.
if REUSE_EXISTING_PROCESSED_DATA:
    _neg_full = None
    _pos_full = None
    print(
        "Source-cohort distribution was not reconstructed: the raw RSNA metadata "
        "is unavailable, and the processed manifest represents only the curated cohort."
    )
else:
    _neg_full = int((df_all["cancer"] == 0).sum())
    _pos_full = int((df_all["cancer"] == 1).sum())
    plot_class_distribution(
        neg_count=_neg_full,
        pos_count=_pos_full,
        title="Stage 1 — complete RSNA source cohort (all projections)",
        subtitle="image level, all views",
        count_unit="images",
        filename="dataset_full_distribution.png",
    )

## 2. Index the source PNG files

The next two cells build deterministic lookup structures that connect metadata identifiers to files. In source-processing mode, all `.png` and `.PNG` files below `DATASET_DIR` are enumerated; in verified local-reuse mode, the index is built exclusively from audited `processed_path` entries. Duplicate path entries are removed while preserving order, and two indices are created: an exact filename-stem map and a token map derived by splitting stems on underscores and hyphens. Tokens that refer to more than one file are removed rather than used ambiguously.

`find_image_path(patient_id, image_id)` then searches, in order, for common direct filenames, exact indexed stems, and a uniquely indexed `image_id` token. It returns a `Path` when a file can be resolved and `None` otherwise.

**Inputs:** the extracted image tree and the configured extensions. **Outputs:** `png_files`, `stem_map`, `token_map`, diagnostic duplicate/ambiguity counts, and the lookup function used later. Execution fails if no PNG is found. Duplicate stems are reported and the first encountered path remains in the exact-stem map; ambiguous tokens are excluded, which avoids silently resolving a non-unique token.


In [ ]:
# Build a deterministic image index from the active data representation.
if REUSE_EXISTING_PROCESSED_DATA:
    png_files = [resolve_path(value) for value in df_all["processed_path"].astype(str)]
    png_files = list(dict.fromkeys(png_files))
    png_count = len(png_files)
    image_index_source = "audited processed_path entries"
else:
    png_files = []
    for ext in IMAGE_EXTENSIONS:
        png_files.extend(DATASET_DIR.rglob(f"*{ext}"))
    png_files = list(dict.fromkeys(png_files))
    png_count = sum(
        len(list(FINAL_DATASET_DIR.rglob(pattern)))
        for pattern in ("*.png", "*.PNG")
    )
    image_index_source = str(DATASET_DIR)

print("Image-index source:", image_index_source)
print("PNG files indexed:", len(png_files))
if len(png_files) == png_count:
    print("No duplicate image paths were found.")
else:
    print("Duplicate image paths:", png_count - len(png_files))
if not png_files:
    raise FileNotFoundError(f"No PNG images are available from {image_index_source}")

# Exact stems resolve before tokens; ambiguous identifiers are removed.
stem_map = {}
duplicate_stems = set()
for path in png_files:
    stem = path.stem
    if stem in stem_map:
        duplicate_stems.add(stem)
    else:
        stem_map[stem] = path

token_map = {}
ambiguous_tokens = set()
for path in png_files:
    stem = path.stem
    tokens = stem.replace("-", "_").split("_")
    tokens.append(stem)
    for token in tokens:
        if token in token_map and token_map[token] != path:
            ambiguous_tokens.add(token)
        else:
            token_map[token] = path
for token in ambiguous_tokens:
    token_map.pop(token, None)

print("Indexed stems:", len(stem_map))
print("Unique indexed tokens:", len(token_map))
print("Duplicate stems (first path retained):", len(duplicate_stems))
print("Ambiguous tokens removed:", len(ambiguous_tokens))

In [ ]:
def find_image_path(patient_id, image_id):
    """
    Cerca l'immagine dentro DATASET_DIR.
    Supporta:
    - image_id.png
    - patient_id_image_id.png
    - file con image_id incluso nel nome
    """
    patient_id = str(patient_id)
    image_id = str(image_id)

    # Candidati diretti più probabili
    for ext in IMAGE_EXTENSIONS:
        candidates = [
            DATASET_DIR / f"{image_id}{ext}",
            DATASET_DIR / f"{patient_id}_{image_id}{ext}",
            DATASET_DIR / f"{image_id}_{patient_id}{ext}",
        ]

        for candidate in candidates:
            if candidate.exists():
                return candidate

    # Mappa su nome senza estensione
    exact_keys = [
        image_id,
        f"{patient_id}_{image_id}",
        f"{image_id}_{patient_id}",
    ]

    for key in exact_keys:
        if key in stem_map:
            return stem_map[key]

    # Fallback: image_id come token unico nel nome file
    if image_id in token_map:
        return token_map[image_id]

    return None

## 3. Restrict the cohort to MLO views and assign patient-level status

Patient status is computed as the maximum `cancer` value across every row for each `patient_id` and merged back into the image table as `patient_label`. The code then retains only rows whose `view` equals the configured `VIEW_FILTER` (`MLO`). Two candidate pools are constructed:

- positive candidates are MLO images with image-level `cancer = 1`;
- negative candidates are MLO images with both `patient_label = 0` and image-level `cancer = 0`.

**Input:** the validated source table. **Outputs:** `df_mlo`, `positive_candidates`, and `negative_candidates`, together with image and patient counts. Requiring `patient_label = 0` for the negative pool prevents a patient with any recorded positive image from contributing a nominally negative image. This is a label-consistency safeguard and defines the patient-level inclusion policy used by the experimental cohort.


In [ ]:
# label dei pazienti

# raggruppa tutte le immagini per paziente e prende il max di cancro
patient_label_df = (
    df_all.groupby("patient_id", as_index=False)["cancer"]
    .max()
    .rename(columns={"cancer": "patient_label"})
)

# Unisce patient_label_df a df_all usando patient_id
df = df_all.merge(patient_label_df, on="patient_id", how="left")

# Manteniamo solo le immagini con la vista MLO
df_mlo = df[df["view"] == VIEW_FILTER].copy().reset_index(drop=True)

print("Immagini MLO:", len(df_mlo))
print("\nDistribuzione cancer nelle MLO:")
print(df_mlo["cancer"].value_counts())

print("\nPazienti nelle MLO:", df_mlo["patient_id"].nunique())
print("Pazienti positivi con almeno una MLO:", int(df_mlo.groupby("patient_id")["patient_label"].max().sum()))

In [ ]:
# Positive: paziente positivo + immagine MLO positiva

positive_candidates = df_mlo[(df_mlo["cancer"] == 1)].copy().reset_index(drop=True)

# Negative: paziente realmente negativo, escludiamo le immagini negative appartenenti a pazienti positivi
# (se presenti)
negative_candidates = df_mlo[(df_mlo["patient_label"] == 0) & (df_mlo["cancer"] == 0)].copy().reset_index(drop=True)

print("Candidati positivi MLO:", len(positive_candidates))
print("Pazienti positivi con MLO positiva:", positive_candidates["patient_id"].nunique())

print("\nCandidati negativi MLO:", len(negative_candidates))
print("Pazienti negativi con MLO:", negative_candidates["patient_id"].nunique())

### 3.1 Class distribution after MLO restriction

The following cell counts the positive and negative candidate images after the view and patient-status rules, but before one-image-per-patient selection and negative subsampling. It is source-dependent and therefore omitted in local-reuse mode, where only the already curated cohort is available. It passes the already computed counts to the common plotting function and saves `dataset_mlo_distribution.png`.

The plot is explicitly image-level: a patient may still contribute more than one candidate at this stage. Its scientific role is to isolate the effect of the MLO and label-consistency filters from the later patient-level sampling policy. No data are modified by this cell.


In [ ]:
if REUSE_EXISTING_PROCESSED_DATA:
    _neg_mlo = None
    _pos_mlo = None
    print(
        "Pre-sampling MLO distribution was not reconstructed from the final processed cohort."
    )
else:
    _neg_mlo = int(len(negative_candidates))
    _pos_mlo = int(len(positive_candidates))
    plot_class_distribution(
        neg_count=_neg_mlo,
        pos_count=_pos_mlo,
        title="Stage 2 — MLO candidates before sampling",
        subtitle="image level, MLO projection only",
        count_unit="images",
        filename="dataset_mlo_distribution.png",
    )

## 4. Select one image per patient and define the working class ratio

`choose_one_per_patient` assigns a seeded random value to each candidate row, sorts by patient and that value, and keeps the first row per `patient_id`. Positive and negative pools use different deterministic seeds (`RANDOM_STATE` and `RANDOM_STATE + 1`) before being concatenated. Assertions verify that `patient_id` remains a column, that every retained patient appears once, and that labels are binary.

The next cell preserves every selected positive and samples at most five negatives per positive with `pandas.DataFrame.sample(random_state=RANDOM_STATE)`. If fewer negatives are available, all available negatives are used; the combined table is then shuffled deterministically. Missing positive or negative classes cause an immediate error.

**Input:** the MLO candidate tables. **Output:** `selected_df`, containing one row per patient and a controlled, data-dependent negative-to-positive ratio no greater than the configured 5:1 target. Scientifically, this step fixes the patient as the independent sampling unit and defines the real cohort composition used in all downstream splits.


In [ ]:
def choose_one_per_patient(input_df, random_state=42):
    """
    Seleziona una sola riga per patient_id in modo riproducibile.
    Non usa groupby.apply, quindi evita problemi in cui patient_id diventa indice.
    """
    
    if "patient_id" not in input_df.columns:
        raise ValueError("La colonna patient_id non è presente nel dataframe di input.")

    rng = np.random.default_rng(random_state)

    out = input_df.copy().reset_index(drop=True)
    out["_random_choice"] = rng.random(len(out))

    out = (
        out.sort_values(["patient_id", "_random_choice"])
        .drop_duplicates(subset=["patient_id"], keep="first")
        .drop(columns=["_random_choice"])
        .reset_index(drop=True)
    )

    return out


# evitiamo che le sequenze di casuali siano le stesse, mantenendo la riproducibilità
positive_one = choose_one_per_patient(positive_candidates, random_state=RANDOM_STATE)
negative_one = choose_one_per_patient(negative_candidates, random_state=RANDOM_STATE + 1)

selected_df = pd.concat([positive_one, negative_one], ignore_index=True)
selected_df = selected_df.reset_index(drop=True)

# Label finale per il progetto
selected_df["label"] = selected_df["cancer"].astype(int)

print("Colonne selected_df:")
print(selected_df.columns.tolist())

print("\nDataset selezionato:", selected_df.shape)
print("\nDistribuzione label:")
print(selected_df["label"].value_counts())

print("\nPazienti unici:", selected_df["patient_id"].nunique())

assert "patient_id" in selected_df.columns, "Errore: patient_id non è una colonna."
assert selected_df["patient_id"].nunique() == len(selected_df), "Errore: ci sono più immagini per lo stesso paziente."
assert set(selected_df["label"].unique()).issubset({0, 1}), "Errore: label non binaria."

display(selected_df.head())

In [ ]:
# Fissiamo un rate di negativi rispetto al numero di positivi

NEGATIVE_TO_POSITIVE_RATIO = 5

positive_df = selected_df[selected_df["label"] == 1].copy()
negative_df = selected_df[selected_df["label"] == 0].copy()

n_pos = len(positive_df)
n_neg_available = len(negative_df)

if n_pos == 0:
    raise ValueError("Non ci sono positivi in selected_df: impossibile fissare il rapporto negativi/positivi.")

if n_neg_available == 0:
    raise ValueError("Non ci sono negativi in selected_df: impossibile fissare il rapporto negativi/positivi.")

if NEGATIVE_TO_POSITIVE_RATIO is None:
    n_neg_target = n_neg_available
else:
    n_neg_target = int(n_pos * NEGATIVE_TO_POSITIVE_RATIO)

    if n_neg_target > n_neg_available:
        print(
            f"Attenzione: richiesti {n_neg_target} negativi, "
            f"ma disponibili solo {n_neg_available}. Verranno usati tutti i negativi disponibili."
        )
        n_neg_target = n_neg_available

negative_sampled_df = negative_df.sample(
    n=n_neg_target,
    random_state=RANDOM_STATE
).copy()

selected_df = pd.concat(
    [positive_df, negative_sampled_df],
    ignore_index=True
)

# Shuffle finale riproducibile
selected_df = selected_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("Dataset dopo definizione rapporto negativi/positivi:")
print(selected_df["label"].value_counts())

actual_n_pos = (selected_df["label"] == 1).sum()
actual_n_neg = (selected_df["label"] == 0).sum()

print(f"\nPositivi: {actual_n_pos}")
print(f"Negativi: {actual_n_neg}")
print(f"Rapporto negativi/positivi effettivo: {actual_n_neg / actual_n_pos:.2f}:1")

assert selected_df["patient_id"].nunique() == len(selected_df), "Errore: ci sono più immagini per lo stesso paziente."
assert selected_df["label"].isin([0, 1]).all(), "Errore: label non binaria."

### 4.1 Curated-cohort distribution and reduction funnel

This cell summarizes `selected_df` after patient-level selection and negative subsampling. In local-reuse mode the final-cohort plot remains valid, but the source-to-final reduction funnel is intentionally omitted because the raw and pre-sampling populations cannot be reconstructed from final artifacts. It saves a class-distribution chart (`dataset_sampled_distribution.png`) and a common-axis funnel (`dataset_reduction_funnel.png`) that juxtaposes negative and positive counts at three stages: the complete source table, the MLO candidate pool, and the final sampled cohort.

Because `selected_df` contains one row per patient, its final image counts also equal patient counts. Earlier stages remain image-level, as stated in the plot labels. The outputs are descriptive figures and printed count transitions; the code does not alter the cohort or estimate uncertainty.


In [ ]:
_neg_final = int((selected_df["label"] == 0).sum())
_pos_final = int((selected_df["label"] == 1).sum())

plot_class_distribution(
    neg_count=_neg_final,
    pos_count=_pos_final,
    title=(
        "Verified canonical processed cohort"
        if REUSE_EXISTING_PROCESSED_DATA
        else "Stage 3 — final cohort after 5:1 sampling"
    ),
    subtitle="one MLO image per patient",
    count_unit="images",
    filename="dataset_sampled_distribution.png",
)

if REUSE_EXISTING_PROCESSED_DATA:
    print(
        "Reduction funnel omitted: source and pre-sampling counts are unavailable "
        "in processed-data reuse mode."
    )
else:
    plot_negative_reduction_funnel(
        stage_labels=[
            "Complete RSNA\n(all views)",
            "MLO filter\n(pre-sampling)",
            "Final cohort\n(5:1 sampling)",
        ],
        neg_values=[_neg_full, _neg_mlo, _neg_final],
        pos_values=[_pos_full, _pos_mlo, _pos_final],
        count_unit="images",
        filename="dataset_reduction_funnel.png",
    )
    print("Negative reduction:", _neg_full, "->", _neg_mlo, "->", _neg_final)
    print("Positive counts by stage:", _pos_full, "->", _pos_mlo, "->", _pos_final)

## 5. Resolve selected metadata rows to source images

Each row in `selected_df` is passed to `find_image_path`, and the resolved path is stored in `image_path`. The lookup targets raw source images in source-processing mode and the audited processed-image index in local-reuse mode. The code reports the number of successful and unsuccessful matches and displays up to 20 unresolved examples for diagnosis. Rows without a resolved file are then removed, paths are converted to strings, and patient uniqueness is asserted again.

**Inputs:** selected `patient_id` and `image_id` values plus the PNG indices created earlier. **Output:** a file-backed `selected_df` ready for splitting. An important operational caveat is that unmatched rows are excluded rather than causing an immediate failure; therefore, the printed post-match class counts must be inspected because file loss can change the realized class ratio.


In [ ]:
image_paths = []

# tqdm mostra barra di avanzamento, .iterarrows restituisce indice e contenuto della riga
for _, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="Ricerca immagini"):  
    path = find_image_path(
        patient_id=row["patient_id"],
        image_id=row["image_id"]
    )
    image_paths.append(path)

selected_df["image_path"] = image_paths

missing_df = selected_df[selected_df["image_path"].isna()].copy()

print("Immagini trovate:", selected_df["image_path"].notna().sum())
print("Immagini mancanti:", selected_df["image_path"].isna().sum())

if len(missing_df) > 0:
    print("\nEsempi immagini mancanti:")
    display(missing_df[["patient_id", "image_id", "view", "cancer"]].head(20))

In [ ]:
# Teniamo solo immagini realmente trovate

selected_df = selected_df[selected_df["image_path"].notna()].copy().reset_index(drop=True)
selected_df["image_path"] = selected_df["image_path"].astype(str)

print("Dataset dopo controllo file:", selected_df.shape)
print(selected_df["label"].value_counts())

assert selected_df["patient_id"].nunique() == len(selected_df), "Errore dopo path matching: più immagini per paziente."

## 6. Create stratified train, validation, and test partitions

In source-processing mode, the partitioning function applies two seeded, label-stratified splits. In local-reuse mode, the published split assignments are preserved exactly and validated rather than recomputed. First, it reserves `VAL_SIZE + TEST_SIZE` of the rows; second, it divides that temporary subset according to the requested validation and test proportions. With the configured values, the intended fractions are 70% training, 15% validation, and 15% test. A `split` column is assigned before the partitions are concatenated into `final_df`.

The function requires both classes and at least two samples in each class before attempting stratification. The input already contains one row per patient, so row-level splitting is also patient-level splitting. The output is `final_df`, and the printed cross-tabulation exposes the realized integer counts after stratification.


In [ ]:
def stratified_train_val_test_split(df, val_size=0.15, test_size=0.15, random_state=42):
    """
    Split train/val/test stratificato.
    Se la stratificazione non è possibile per pochi campioni, solleva un errore chiaro.
    """
    
    label_counts = df["label"].value_counts()
    if len(label_counts) < 2:
        raise ValueError("Serve almeno una classe positiva e una negativa per fare split stratificato.")

    if (label_counts < 2).any():
        raise ValueError(
            "Almeno una classe ha meno di 2 campioni. "
            "Impossibile fare split stratificato affidabile."
        )

    train_df, temp_df = train_test_split(
        df,
        test_size=val_size + test_size,
        stratify=df["label"],
        random_state=random_state
    )

    relative_test_size = test_size / (val_size + test_size)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test_size,
        stratify=temp_df["label"],
        random_state=random_state
    )

    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"

    final_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    return final_df


if REUSE_EXISTING_PROCESSED_DATA:
    if "split" not in selected_df.columns:
        raise ValueError("The verified processed manifest does not contain split assignments.")
    final_df = selected_df.copy().reset_index(drop=True)
    if set(final_df["split"].astype(str).unique()) != set(EDA_SPLIT_ORDER):
        raise ValueError("Published split assignments are incomplete.")
    print("Preserved published train/validation/test assignments.")
else:
    final_df = stratified_train_val_test_split(
        selected_df,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

print("Distribuzione split/label:")
print(pd.crosstab(final_df["split"], final_df["label"]))

print("\nNumero righe final_df:", len(final_df))
print("Pazienti unici final_df:", final_df["patient_id"].nunique())

### 6.1 Partition composition and patient-leakage audit

The EDA cell reads the current `final_df` when available, otherwise it falls back to the persisted all-samples manifest, and constructs a split-by-label count table. It saves `class_balance_per_split.png` with annotated negative and positive counts for the training, validation, and test partitions.

The subsequent audit converts patient identifiers in each split to sets and asserts pairwise disjointness, then confirms once more that the total number of rows equals the number of unique patients. These are hard leakage safeguards: execution stops if any patient crosses a partition boundary or contributes more than one row. The plot is descriptive; the assertions establish the independence policy required for downstream validation and test analyses.


In [ ]:
# EDA split/classi: sbilanciamento positive vs negative nei tre split (train/val/test)
# Sola lettura dei metadati gia' prodotti, nessun preprocessing ricalcolato.
eda_split_df, eda_split_source = get_split_dataframe_for_eda()

if eda_split_df is not None:
    eda_split_df["split"] = eda_split_df["split"].astype(str)
    eda_split_df["label"] = eda_split_df["label"].astype(int)
    print("Sorgente EDA split/classi:", eda_split_source)

    balance = (
        eda_split_df.groupby(["split", "label"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=EDA_SPLIT_ORDER, fill_value=0)
        .reindex(columns=EDA_LABEL_ORDER, fill_value=0)
    )

    x_positions = np.arange(len(EDA_SPLIT_ORDER))
    width = 0.36
    fig, ax = plt.subplots(figsize=(9, 5))
    negative_bars = ax.bar(
        x_positions - width / 2, balance[0].to_numpy(), width,
        label="negative", color=EDA_LABEL_COLORS[0],
    )
    positive_bars = ax.bar(
        x_positions + width / 2, balance[1].to_numpy(), width,
        label="positive", color=EDA_LABEL_COLORS[1],
    )
    set_annotated_bar_ylim(ax, balance.to_numpy())
    annotate_bars(ax, negative_bars)
    annotate_bars(ax, positive_bars)
    ax.set_title("Sbilanciamento di classe per split (dataset finale)")
    ax.set_xlabel("Split")
    ax.set_ylabel("Numero di immagini")
    ax.set_xticks(x_positions)
    ax.set_xticklabels(EDA_SPLIT_ORDER)
    ax.legend(title="Classe")
    ax.grid(axis="y", alpha=0.25)
    show_and_save_preprocessing_plot(fig, "class_balance_per_split.png")

In [ ]:
# Controllo anti-leakage
train_patients = set(final_df[final_df["split"] == "train"]["patient_id"])
val_patients = set(final_df[final_df["split"] == "val"]["patient_id"])
test_patients = set(final_df[final_df["split"] == "test"]["patient_id"])

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

assert final_df["patient_id"].nunique() == len(final_df)

print("Nessun patient leakage. Un solo campione per paziente.")

## 7. Standardize image representation

Three functions implement the image transformation. `load_as_grayscale_uint8` converts multichannel images to PIL mode `L`, passes through existing 8-bit grayscale data, and robustly rescales higher-bit-depth or other numeric arrays using the 0.5th and 99.5th percentiles of non-zero pixels. If no non-zero pixels exist, the full range is considered; if the resulting range is degenerate, a zero-valued image is returned.

`resize_with_padding` applies Lanczos resampling while preserving aspect ratio, centers the resized image on a black square canvas, and performs no crop. `preprocess_image` creates the destination directory and saves the result.

**Input:** a resolved source image. **Output:** a single-channel, 8-bit, 512 × 512 PNG. This normalization controls image size and storage representation while retaining the complete resized field of view. The percentile operation is an intensity standardization rule, not a diagnostic enhancement claim.


In [ ]:
def load_as_grayscale_uint8(path):
    """
    Carica un'immagine e la restituisce come PIL grayscale 8-bit.

    Se l'immagine è 16-bit o ha valori molto ampi, applica una normalizzazione robusta
    usando i percentili 0.5 e 99.5 dei pixel non-zero.
    """
    img = Image.open(path)
    arr = np.array(img)

    # Caso RGB/RGBA: conversione PIL standard.
    if arr.ndim == 3:
        return img.convert("L")

    # Caso già uint8 (unsigned integer 8-bit)
    if arr.dtype == np.uint8:
        return Image.fromarray(arr, mode="L")

    # Caso 16-bit o altro: normalizzazione robusta.
    arr = arr.astype(np.float32)

    nonzero = arr[arr > 0]

    if len(nonzero) > 0:
        lo, hi = np.percentile(nonzero, [0.5, 99.5])
    else:
        lo, hi = arr.min(), arr.max()

    if hi <= lo:
        arr_norm = np.zeros_like(arr, dtype=np.uint8)
    else:
        arr = np.clip(arr, lo, hi)
        arr_norm = ((arr - lo) / (hi - lo) * 255.0).astype(np.uint8)

    return Image.fromarray(arr_norm, mode="L")


def resize_with_padding(img, size=512):
    """
    Resize mantenendo aspect ratio + padding nero.
    """
    img = img.copy()

    if img.size == (size, size):
        return img

    img.thumbnail((size, size), Image.Resampling.LANCZOS)

    canvas = Image.new("L", (size, size), color=0)

    left = (size - img.width) // 2
    top = (size - img.height) // 2

    canvas.paste(img, (left, top))

    return canvas


def preprocess_image(input_path, output_path, size=512):
    input_path = Path(input_path)
    output_path = Path(output_path)

    output_path.parent.mkdir(parents=True, exist_ok=True)

    img = load_as_grayscale_uint8(input_path)
    img = resize_with_padding(img, size=size)

    img.save(output_path)

## 8. Materialize the processed dataset

The output hierarchy `data/processed/<split>/<label>/` and its metadata directory are created before image conversion. If `RESET_OUTPUT_DIR` is explicitly enabled, the entire processed-data directory is removed first; it is disabled by default. In source-processing mode, each row of `final_df` is transformed with the functions above and saved under a filename containing patient, image, laterality, and view identifiers. In verified local-reuse mode, image conversion is skipped and the audited manifest is loaded read-only; this prevents already normalized images from being treated as raw inputs or overwritten.

A sustainability context records a genuine image-conversion run. No conversion or emissions record is fabricated for a read-only reuse pass. Successful conversions append a source-metadata row to `processed_rows`, including source identifiers, split, label, and paths made relative to the repository when possible. Per-image exceptions are printed and skipped rather than re-raised.

**Inputs:** `final_df` and its resolved source paths. **Outputs:** processed PNG files and the in-memory `processed_rows` manifest. Directory creation and explicit reset control protect existing data, while the later count summaries make any skipped conversion visible. Because failures are non-fatal here, the number of successful outputs must be compared with the number of input rows.


In [ ]:
if RESET_OUTPUT_DIR and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ["train", "val", "test"]:
    for label in [0, 1]:
        (OUTPUT_DIR / split / str(label)).mkdir(parents=True, exist_ok=True)

(OUTPUT_DIR / "metadata").mkdir(parents=True, exist_ok=True)
for directory in [PLOTS_DIR, METRICS_DIR, ECOTRACKER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Cartella output pronta:", OUTPUT_DIR)

In [ ]:
def to_relative(path, base):
    """Return a project-relative path, or a non-absolute filename as fallback."""
    path = Path(path).resolve()
    try:
        return str(path.relative_to(base))
    except ValueError:
        return path.name


processed_rows = []
eco_preprocessing = None

if REUSE_EXISTING_PROCESSED_DATA:
    processed_rows = pd.read_csv(PROCESSED_REUSE_MANIFEST).to_dict(orient="records")
    print(
        f"Read-only reuse accepted for {len(processed_rows)} processed rows; "
        "image conversion was not repeated."
    )
else:
    with measure_sustainability(
        label="preprocessing_images", sample_interval=0.1
    ) as eco_preprocessing:
        for _, row in tqdm(
            final_df.iterrows(), total=len(final_df), desc="Preprocessing images"
        ):
            patient_id = str(row["patient_id"])
            image_id = str(row["image_id"])
            laterality = str(row["laterality"])
            view = str(row["view"])
            label = int(row["label"])
            split = str(row["split"])

            input_path = resolve_path(row["image_path"])
            filename = f"{patient_id}_{image_id}_{laterality}_{view}.png"
            output_path = OUTPUT_DIR / split / str(label) / filename

            try:
                preprocess_image(input_path=input_path, output_path=output_path, size=IMG_SIZE)
                processed_rows.append({
                    "patient_id": patient_id,
                    "image_id": image_id,
                    "laterality": laterality,
                    "view": view,
                    "label": label,
                    "cancer": int(row["cancer"]),
                    "patient_label": int(row["patient_label"]),
                    "split": split,
                    "source": "real",
                    "original_path": to_relative(input_path, BASE_DIR),
                    "processed_path": to_relative(output_path, BASE_DIR),
                })
            except Exception as exc:
                print(f"Image conversion failed for {input_path}: {exc}")

## 9. Persist manifests and preprocessing metadata

`processed_rows` is converted to a DataFrame and partitioned by the recorded `split`. In source-processing mode, four CSV manifests—`all_processed.csv`, `train.csv`, `val.csv`, and `test.csv`—are written to `data/processed/metadata/`. In local-reuse mode, these already reconciled manifests are loaded into memory but deliberately not rewritten. These manifests are the primary interface consumed by later notebooks.

When sustainability metrics from an actual conversion are available from the preceding context, they are augmented with a timestamp, notebook identity, input/output row counts, output directory, and completion status, then saved to `preprocessing_ecotracker.json`. If that context is unavailable, the cell reports that the image-processing cell must be rerun. Printed totals, class counts, and the split-by-label table provide a direct audit of the persisted cohort.

The scientific role of these outputs is reproducibility: image files, cohort membership, and resource-use metadata are recorded separately so downstream stages can trace each processed sample without re-reading the original archive.


In [ ]:
processed_df = pd.DataFrame(processed_rows)

train_processed = processed_df[processed_df["split"] == "train"].reset_index(drop=True)
val_processed = processed_df[processed_df["split"] == "val"].reset_index(drop=True)
test_processed = processed_df[processed_df["split"] == "test"].reset_index(drop=True)

if REUSE_EXISTING_PROCESSED_DATA:
    print("Verified manifests remain unchanged during read-only reuse.")
else:
    processed_df.to_csv(OUTPUT_DIR / "metadata" / "all_processed.csv", index=False)
    train_processed.to_csv(OUTPUT_DIR / "metadata" / "train.csv", index=False)
    val_processed.to_csv(OUTPUT_DIR / "metadata" / "val.csv", index=False)
    test_processed.to_csv(OUTPUT_DIR / "metadata" / "test.csv", index=False)

    if eco_preprocessing is not None and getattr(eco_preprocessing, "metrics", None) is not None:
        preprocessing_eco_record = eco_preprocessing.metrics.to_dict()
        preprocessing_eco_record.update({
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "record_type": "preprocessing_images",
            "notebook": "01_Preprocessing_RSNA_512_gray_MLO",
            "output_dir": str(OUTPUT_DIR),
            "n_input_rows": int(len(final_df)),
            "n_processed_rows": int(len(processed_df)),
            "status": "completed",
        })
        with PREPROCESSING_ECOTRACKER_PATH.open("w", encoding="utf-8") as handle:
            json.dump(preprocessing_eco_record, handle, indent=2, ensure_ascii=False)
        print("EcoTracker JSON:", PREPROCESSING_ECOTRACKER_PATH)
    else:
        print("No preprocessing sustainability metrics were produced.")

print("Processed cohort available.")
print("Total:", len(processed_df))
print("Train:", len(train_processed))
print("Validation:", len(val_processed))
print("Test:", len(test_processed))
print("\nOverall label distribution:")
print(processed_df["label"].value_counts())
print("\nSplit-by-label distribution:")
print(pd.crosstab(processed_df["split"], processed_df["label"]))
display(processed_df.head())

## 10. Normalize the visual tissue orientation

In source-processing mode, the orientation pass standardizes breast tissue to the left side of the image. In local-reuse mode, every image is inspected read-only and execution stops if any current image is right-oriented; the historical orientation columns and image bytes are preserved. For each processed PNG, `detect_tissue_side` thresholds intensities above 5, weights the remaining foreground by pixel intensity, and compares the summed mass in the left and right halves. Images classified as right-sided are mirrored horizontally in place; left-sided images are retained; images with no foreground mass are marked `unknown` and are not flipped.

The code records the pre- and post-operation side, left/right intensity ratios, the flip decision, and the configured target side in every metadata row. It also assigns `normalized_laterality = "L"` as a project convention; this field is not inferred from the original laterality annotation and should not be interpreted as anatomical laterality after mirroring. Updated split manifests overwrite the preceding versions only after a source-processing run; verified reuse never rewrites them.

**Inputs:** processed PNGs and `all_processed.csv`. **Outputs:** potentially mirrored PNGs, enriched CSV manifests, and `preprocessing_evaluation.json` with cohort and orientation counts. Missing files or required columns stop execution. A final assertion forbids any post-normalization value of `right`; `unknown` remains permitted and is reported explicitly. This step standardizes visual presentation for later models while preserving an audit trail of every modification.


In [ ]:
TARGET_TISSUE_SIDE = "left"
metadata_path = OUTPUT_DIR / "metadata" / "all_processed.csv"
if not metadata_path.exists():
    raise FileNotFoundError(f"Processed metadata not found: {metadata_path}")

processed_df = pd.read_csv(metadata_path)
required_cols = ["processed_path", "laterality", "split", "label"]
for column in required_cols:
    if column not in processed_df.columns:
        raise ValueError(f"Required processed-metadata column is missing: {column}")


def detect_tissue_side(image_path, min_threshold=5):
    """Classify the dominant tissue side from foreground-weighted intensity mass."""
    with Image.open(image_path) as image:
        array = np.array(image.convert("L"))
    _, width = array.shape
    foreground = np.where(array > min_threshold, array.astype(np.float64), 0.0)
    if foreground.sum() == 0:
        return "unknown", 0.0, 0.0
    left_mass = foreground[:, : width // 2].sum()
    right_mass = foreground[:, width // 2 :].sum()
    total_mass = left_mass + right_mass
    left_ratio = left_mass / total_mass
    right_ratio = right_mass / total_mass
    side = "left" if left_mass >= right_mass else "right"
    return side, left_ratio, right_ratio


if REUSE_EXISTING_PROCESSED_DATA:
    orientation_columns = {
        "visual_side_before", "visual_side_after", "left_ratio_before",
        "right_ratio_before", "flipped_by_visual_rule",
        "normalized_tissue_side", "normalized_laterality",
    }
    missing_orientation_columns = sorted(orientation_columns.difference(processed_df.columns))
    if missing_orientation_columns:
        raise ValueError(
            "Verified processed metadata lacks orientation information: "
            + ", ".join(missing_orientation_columns)
        )

    visual_side_before = processed_df["visual_side_before"].astype(str).tolist()
    left_ratio_before = processed_df["left_ratio_before"].astype(float).tolist()
    right_ratio_before = processed_df["right_ratio_before"].astype(float).tolist()
    flipped_by_visual_rule = processed_df["flipped_by_visual_rule"].astype(bool).tolist()
    visual_side_after = []
    for image_reference in tqdm(
        processed_df["processed_path"], desc="Validating normalized orientation"
    ):
        image_path = resolve_path(image_reference)
        if not image_path.is_file():
            raise FileNotFoundError(f"Processed image not found: {image_path}")
        side_after, _, _ = detect_tissue_side(image_path)
        visual_side_after.append(side_after)

    if any(side == "right" for side in visual_side_after):
        raise AssertionError(
            "Read-only orientation validation found images with right-sided tissue."
        )
    flipped_count = int(sum(flipped_by_visual_rule))
    unknown_count = int(sum(side == "unknown" for side in visual_side_after))
    kept_count = int(len(processed_df) - flipped_count - unknown_count)
    print("Read-only orientation validation completed; no image bytes were modified.")
else:
    visual_side_before = []
    visual_side_after = []
    left_ratio_before = []
    right_ratio_before = []
    flipped_by_visual_rule = []
    flipped_count = 0
    kept_count = 0
    unknown_count = 0

    for _, row in tqdm(processed_df.iterrows(), total=len(processed_df), desc="Normalizing orientation"):
        image_path = resolve_path(row["processed_path"])
        if not image_path.exists():
            raise FileNotFoundError(f"Processed image not found: {image_path}")
        side_before, left_ratio, right_ratio = detect_tissue_side(image_path)
        visual_side_before.append(side_before)
        left_ratio_before.append(left_ratio)
        right_ratio_before.append(right_ratio)
        should_flip = (
            (TARGET_TISSUE_SIDE == "left" and side_before == "right")
            or (TARGET_TISSUE_SIDE == "right" and side_before == "left")
        )
        if should_flip:
            with Image.open(image_path) as image:
                ImageOps.mirror(image.convert("L")).save(image_path)
            flipped_count += 1
            flipped_by_visual_rule.append(True)
        elif side_before == "unknown":
            unknown_count += 1
            flipped_by_visual_rule.append(False)
        else:
            kept_count += 1
            flipped_by_visual_rule.append(False)
        side_after, _, _ = detect_tissue_side(image_path)
        visual_side_after.append(side_after)

    processed_df["visual_side_before"] = visual_side_before
    processed_df["visual_side_after"] = visual_side_after
    processed_df["left_ratio_before"] = left_ratio_before
    processed_df["right_ratio_before"] = right_ratio_before
    processed_df["flipped_by_visual_rule"] = flipped_by_visual_rule
    processed_df["normalized_tissue_side"] = TARGET_TISSUE_SIDE
    processed_df["normalized_laterality"] = "L"

    train_processed = processed_df[processed_df["split"] == "train"].reset_index(drop=True)
    val_processed = processed_df[processed_df["split"] == "val"].reset_index(drop=True)
    test_processed = processed_df[processed_df["split"] == "test"].reset_index(drop=True)
    processed_df.to_csv(OUTPUT_DIR / "metadata" / "all_processed.csv", index=False)
    train_processed.to_csv(OUTPUT_DIR / "metadata" / "train.csv", index=False)
    val_processed.to_csv(OUTPUT_DIR / "metadata" / "val.csv", index=False)
    test_processed.to_csv(OUTPUT_DIR / "metadata" / "test.csv", index=False)

print("Orientation stage completed.")
print("Historically mirrored images:", flipped_count)
print("Images retained without mirroring:", kept_count)
print("Unknown-side images:", unknown_count)
print("\nCurrent visual-side distribution:")
print(pd.Series(visual_side_after).value_counts())
assert all(side != "right" for side in visual_side_after), (
    "Some images remain right-oriented after the orientation stage."
)

split_label_counts = (
    processed_df.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=EDA_SPLIT_ORDER, fill_value=0)
    .reindex(columns=EDA_LABEL_ORDER, fill_value=0)
)
preprocessing_evaluation = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "notebook": "01_Preprocessing_RSNA_512_gray_MLO",
    "data_source_mode": (
        "verified_processed_reuse" if REUSE_EXISTING_PROCESSED_DATA else "raw_preprocessing"
    ),
    "dataset_dir": None if REUSE_EXISTING_PROCESSED_DATA else str(DATASET_DIR),
    "processed_manifest": str(metadata_path),
    "output_dir": str(OUTPUT_DIR),
    "plots_dir": str(PLOTS_DIR),
    "ecotracker_json": str(PREPROCESSING_ECOTRACKER_PATH),
    "img_size": IMG_SIZE,
    "view_filter": VIEW_FILTER,
    "n_rows": int(len(processed_df)),
    "n_patients": int(processed_df["patient_id"].nunique()),
    "label_counts": {
        str(int(label)): int(count)
        for label, count in processed_df["label"].value_counts().sort_index().items()
    },
    "split_label_counts": {
        str(split): {
            str(int(label)): int(split_label_counts.loc[split, label])
            for label in EDA_LABEL_ORDER
        }
        for split in EDA_SPLIT_ORDER
    },
    "visual_normalization": {
        "mode": "read_only_validation" if REUSE_EXISTING_PROCESSED_DATA else "normalization",
        "target_tissue_side": TARGET_TISSUE_SIDE,
        "flipped_count": int(flipped_count),
        "kept_count": int(kept_count),
        "unknown_count": int(unknown_count),
        "after_counts": {
            str(side): int(count)
            for side, count in pd.Series(visual_side_after).value_counts().items()
        },
    },
}
with PREPROCESSING_EVALUATION_PATH.open("w", encoding="utf-8") as handle:
    json.dump(preprocessing_evaluation, handle, indent=2, ensure_ascii=False)
print("Evaluation JSON:", PREPROCESSING_EVALUATION_PATH)

### 10.1 Qualitative audit of processed images

The final EDA cell loads the current processed manifest, preferentially from memory and otherwise from disk. It samples up to six images per class with fixed seeds, fills any remaining positions from the unused pool, shuffles the display order reproducibly, and excludes missing files while reporting their count.

Up to 12 valid grayscale images are arranged in a 3 × 4 panel labeled by class, split, and filename. The figure is saved as `sample_grid.png`, displayed, and closed. This is a qualitative integrity check for gross orientation, contrast, and file-readability issues; it does not replace quantitative image-quality assessment or establish clinical validity.


In [ ]:
# EDA visiva: griglia campioni, sola lettura delle immagini preprocessate
eda_processed_df, eda_processed_source = get_processed_dataframe_for_eda()

if eda_processed_df is not None:
    eda_processed_df["split"] = eda_processed_df["split"].astype(str)
    eda_processed_df["label"] = eda_processed_df["label"].astype(int)
    print("Sorgente EDA immagini:", eda_processed_source)

    n_grid_images = 12
    n_per_label = n_grid_images // 2
    sampled_parts = []

    for label in EDA_LABEL_ORDER:
        subset = eda_processed_df[eda_processed_df["label"] == label]
        n_take = min(n_per_label, len(subset))
        if n_take > 0:
            sampled_parts.append(subset.sample(n=n_take, random_state=RANDOM_STATE + label))

    if sampled_parts:
        sample_grid_df = pd.concat(sampled_parts, axis=0)
        remaining = n_grid_images - len(sample_grid_df)
        if remaining > 0:
            remaining_pool = eda_processed_df.drop(index=sample_grid_df.index, errors="ignore")
            n_take = min(remaining, len(remaining_pool))
            if n_take > 0:
                sample_grid_df = pd.concat(
                    [
                        sample_grid_df,
                        remaining_pool.sample(n=n_take, random_state=RANDOM_STATE + 10),
                    ],
                    axis=0,
                )
        sample_grid_df = sample_grid_df.sample(frac=1, random_state=RANDOM_STATE + 20)

        valid_rows = []
        missing_images = 0
        for _, row in sample_grid_df.iterrows():
            image_path = resolve_path(row["processed_path"])
            if image_path.is_file():
                valid_rows.append(row)
            else:
                missing_images += 1

        if missing_images:
            print(f"Campioni esclusi dalla griglia perché mancanti: {missing_images}")

        if valid_rows:
            fig, axes = plt.subplots(3, 4, figsize=(12, 9))
            axes = axes.ravel()
            for ax, row in zip(axes, valid_rows):
                image_array, image_path = read_grayscale_image_from_row(row)
                ax.imshow(image_array, cmap="gray")
                ax.set_title(
                    f'{EDA_LABEL_NAMES[int(row["label"])]} | {row["split"]}\n{image_path.name}',
                    fontsize=9,
                )
                ax.axis("off")
            for ax in axes[len(valid_rows):]:
                ax.axis("off")
            fig.suptitle("Campioni preprocessati MLO 512x512", fontsize=14)
            fig.tight_layout()
            show_and_save_preprocessing_plot(fig, "sample_grid.png")
        else:
            print("Griglia campioni non generata: nessuna immagine valida trovata.")
    else:
        print("Griglia campioni non generata: CSV senza campioni disponibili.")